# IPC Protocol (020)
Demonstrate the length-prefixed JSON protocol and request handling.

In [6]:
%load_ext autoreload
%autoreload 2
import os
import tempfile
from pathlib import Path

from ciphercache.daemon.state import DaemonConfig, DaemonState
from ciphercache.ipc.handler import handle_request
from ciphercache.ipc.framing import decode_single_frame, encode_message
from ciphercache.ttl import parse_ttl

In [8]:
data_dir = Path(tempfile.mkdtemp(prefix="ciphercache-demo-"))
os.chmod(data_dir, 0o700)
config = DaemonConfig(data_dir=data_dir)
state = DaemonState(config=config)
state.unlock(parse_ttl("1h"))
state.secrets["default"] = {"service/api": {"api_key": "demo"}}
ticket_path = state.issue_ticket("demo_client")
print(ticket_path)
ticket = ticket_path.read_text(encoding="utf-8").strip()
print(ticket)

/var/folders/0t/w9l_5c597rdglh2kbffq18sr0000gn/T/ciphercache-demo-_jgt3kul/tickets/demo_client.ticket
wuIbShrHV5V56OGXkjUkvF0qbaN1hSgeCWadODQ8P4E


In [ ]:
ttl_examples = ["5s", "1h 30m", "infinity"]
{value: parse_ttl(value) for value in ttl_examples}


In [ ]:
message = {"version": "v0", "id": "frame-1", "type": "request", "op": "ping", "payload": {}}
frame = encode_message(message)
decoded = decode_single_frame(frame)
{"frame_len": len(frame), "roundtrip_ok": decoded == message, "decoded": decoded}


In [13]:
request = {
    "version": "v0",
    "id": "demo-1",
    "type": "request",
    "op": "get_secret",
    "payload": {
        "ticket": ticket,
        "secret_name": "service/api",
        "store": "default",
    },
}
handle_request(state, request)

{'version': 'v0',
 'id': 'demo-1',
 'type': 'response',
 'op': 'get_secret',
 'payload': {'secret': {'api_key': 'demo'}}}